# Broad 8-K category × five-strategy discovery

This separate notebook preserves the starter. The screen covers 24 categories and five strategies (120 primary comparisons); ten categories meet the preliminary 40-filing gate. Raw filing counts are not usable matched trade counts.

Primary: 21 sessions, 3–6-month expiry, ATM long call and starter 5% OTM overlay legs. All fixed horizons and 3%/5%/10% strikes are retained. Overlays are incremental over stock. Gross marks, costs, capacity, and strict matched inference are distinct stages.


In [1]:
from pathlib import Path
import json
import pandas as pd
OUT = Path('broad_strategy_results')
display(pd.read_csv(OUT/'category_inventory.csv'))
display(pd.read_csv(OUT/'full_calendar_control_coverage_summary.csv'))


,tag,raw_rows,distinct_filings,companies,eligible_for_discovery
0,acquisition_completion,24,24,17,False
1,annual_meeting_results,150,150,87,True
2,business_line_exit,11,11,7,False
3,ceo_appointment,17,17,15,False
4,ceo_departure,31,31,25,False
5,cfo_appointment,34,34,29,False
6,cfo_departure,31,31,24,False
7,cybersecurity_incident,4,4,4,False
8,debt_issuance,277,277,80,True
9,director_appointment,97,97,63,True


,tag,events,events_with_candidate,candidate_date_links
0,annual_meeting_results,150,99,330
1,debt_issuance,277,160,478
2,director_appointment,97,53,167
3,director_departure,101,49,184
4,executive_compensation_change,174,103,336
5,executive_officer_appointment,105,62,227
6,guidance_issuance_or_update,60,30,103
7,quarterly_earnings,131,69,222
8,shareholder_proposal_outcome,121,81,252
9,underwriting_agreement,146,92,268


## Recorded choices

Choices are stored before performance review. Historical validation is not a pristine sealed window. The initial random-placebo pool is preserved separately from the expanded full-calendar pool. Control expansion follows date coverage, not favorable returns.


In [2]:
for name in ['discovery_registration.json', 'cost_specification.json', 'matching_inference_registration.json', 'full_calendar_control_registration.json']:
    print(name)
    print(json.dumps(json.loads((OUT/name).read_text(encoding='utf-8')), indent=2))


discovery_registration.json
{
  "source_sha256": "d42d27119d4395381f82a06a58520295e2908e1146d2a579f02953049381529d",
  "strategies": [
    "long_call",
    "covered_call",
    "protective_put",
    "collar",
    "cash_secured_put"
  ],
  "primary_horizon": 21,
  "expiry_bucket": "3\u20136 months",
  "strikes": "starter rules",
  "categories": [
    "acquisition_completion",
    "annual_meeting_results",
    "business_line_exit",
    "ceo_appointment",
    "ceo_departure",
    "cfo_appointment",
    "cfo_departure",
    "cybersecurity_incident",
    "debt_issuance",
    "director_appointment",
    "director_departure",
    "equity_compensation_grant",
    "executive_compensation_change",
    "executive_officer_appointment",
    "facility_closure",
    "guidance_issuance_or_update",
    "quarterly_earnings",
    "restructuring_plan",
    "share_repurchase_program",
    "shareholder_proposal_outcome",
    "significant_contract_award",
    "strategic_initiative",
    "underwriting_agreemen

## Data collection

Collection uses the API key already loaded through the starter configuration. Run only one pricing process at a time. These flags default to False so opening the notebook shows saved evidence without duplicating an active terminal job.


In [3]:
RUN_COLLECTION = False
RUN_EXPANDED_CONTROLS = False
if RUN_COLLECTION:
    import broad_strategy_screen as screen
    screen.price_all()
if RUN_EXPANDED_CONTROLS:
    import broad_control_inventory as controls
    controls.price_controls()
print('Completed category collections:', [p.parent.name for p in OUT.glob('*/pricing_complete.json')])


Completed category collections: ['annual_meeting_results', 'debt_issuance', 'director_appointment', 'director_departure', 'executive_compensation_change', 'executive_officer_appointment', 'guidance_issuance_or_update', 'quarterly_earnings', 'shareholder_proposal_outcome', 'underwriting_agreement']


## Accounting and saved analysis

Verification checks strategy identities and output coverage. It does not prove fills or statistical significance. Strict controls use the same company/year/quarter/weekday, comparable expiry, and disclosure-free dates. Connected dependence components account conservatively for companies and overlapping holds; too few independent components prevents a reliable interval. Insignificance does not establish no effect.


In [4]:
import verify_broad_strategy_data as verification
verification.verify()
for name in ['strict_matched_summary.csv', 'expanded_strict_matched_summary.csv', 'observed_expanded_strict_matched_summary.csv']:
    path = OUT/name
    if path.exists():
        frame = pd.read_csv(path)
        if not frame.empty:
            primary = frame[(frame.horizon.astype(str)=='21') & (frame.otm==.05) & (frame.cost==.05)]
            print(name)
            compact = primary[['tag', 'strategy', 'events', 'companies', 'dependence_clusters', 'difference', 'ci_lo', 'ci_hi', 'conclusion']].copy()
            compact[['difference', 'ci_lo', 'ci_hi']] *= 100
            compact = compact.rename(columns={'difference': 'difference_pp', 'ci_lo': 'ci_lo_pp', 'ci_hi': 'ci_hi_pp'})
            display(compact)
            print('Differences are percentage points of entry stock notional; missing intervals do not imply no effect.')
    else:
        print(name, 'not yet completed')


PASS annual_meeting_results/events_cost_sensitivity.csv.gz: 81,960 rows; 76,076 finite net rows
PASS annual_meeting_results/ordinary_cost_sensitivity.csv.gz: 63,000 rows; 59,484 finite net rows
PASS debt_issuance/events_cost_sensitivity.csv.gz: 149,160 rows; 139,292 finite net rows
PASS debt_issuance/ordinary_cost_sensitivity.csv.gz: 62,700 rows; 57,908 finite net rows
PASS director_appointment/events_cost_sensitivity.csv.gz: 52,200 rows; 49,768 finite net rows
PASS director_appointment/ordinary_cost_sensitivity.csv.gz: 68,160 rows; 63,908 finite net rows
PASS director_departure/events_cost_sensitivity.csv.gz: 54,420 rows; 51,364 finite net rows
PASS director_departure/ordinary_cost_sensitivity.csv.gz: 66,000 rows; 62,840 finite net rows
PASS executive_compensation_change/events_cost_sensitivity.csv.gz: 94,680 rows; 88,004 finite net rows
PASS executive_compensation_change/ordinary_cost_sensitivity.csv.gz: 65,340 rows; 60,632 finite net rows
PASS executive_officer_appointment/events_co

,tag,strategy,events,companies,dependence_clusters,difference_pp,ci_lo_pp,ci_hi_pp,conclusion
382,annual_meeting_results,long_call,1,1,1,-8.190077,NaN,NaN,INCONCLUSIVE
386,annual_meeting_results,covered_call,1,1,1,5.852343,NaN,NaN,INCONCLUSIVE
390,annual_meeting_results,protective_put,1,1,1,7.967583,NaN,NaN,INCONCLUSIVE
394,annual_meeting_results,collar,1,1,1,13.819926,NaN,NaN,INCONCLUSIVE
398,annual_meeting_results,cash_secured_put,1,1,1,-7.333831,NaN,NaN,INCONCLUSIVE
874,director_appointment,long_call,1,1,1,12.582488,NaN,NaN,INCONCLUSIVE
878,director_appointment,covered_call,1,1,1,-9.870575,NaN,NaN,INCONCLUSIVE
882,director_appointment,protective_put,1,1,1,-4.546037,NaN,NaN,INCONCLUSIVE
886,director_appointment,collar,1,1,1,-14.416612,NaN,NaN,INCONCLUSIVE
890,director_appointment,cash_secured_put,1,1,1,4.839745,NaN,NaN,INCONCLUSIVE


,tag,strategy,events,companies,dependence_clusters,difference_pp,ci_lo_pp,ci_hi_pp,conclusion
382,annual_meeting_results,long_call,56,42,1,-0.278391,NaN,NaN,INCONCLUSIVE
386,annual_meeting_results,covered_call,54,41,1,0.178107,NaN,NaN,INCONCLUSIVE
390,annual_meeting_results,protective_put,52,40,1,0.731222,NaN,NaN,INCONCLUSIVE
394,annual_meeting_results,collar,52,40,1,0.890864,NaN,NaN,INCONCLUSIVE
398,annual_meeting_results,cash_secured_put,55,41,1,-0.739553,NaN,NaN,INCONCLUSIVE
982,debt_issuance,long_call,73,44,1,-0.728288,NaN,NaN,INCONCLUSIVE
986,debt_issuance,covered_call,71,42,1,0.696435,NaN,NaN,INCONCLUSIVE
990,debt_issuance,protective_put,70,42,1,0.101427,NaN,NaN,INCONCLUSIVE
994,debt_issuance,collar,70,42,1,0.895939,NaN,NaN,INCONCLUSIVE
998,debt_issuance,cash_secured_put,73,44,1,-0.073052,NaN,NaN,INCONCLUSIVE


,tag,strategy,events,companies,dependence_clusters,difference_pp,ci_lo_pp,ci_hi_pp,conclusion
382,annual_meeting_results,long_call,56,42,1,-0.277639,NaN,NaN,INCONCLUSIVE
386,annual_meeting_results,covered_call,54,41,1,0.175419,NaN,NaN,INCONCLUSIVE
390,annual_meeting_results,protective_put,52,40,1,0.726255,NaN,NaN,INCONCLUSIVE
394,annual_meeting_results,collar,52,40,1,0.882384,NaN,NaN,INCONCLUSIVE
398,annual_meeting_results,cash_secured_put,55,41,1,-0.734975,NaN,NaN,INCONCLUSIVE
982,debt_issuance,long_call,73,44,1,-0.730094,NaN,NaN,INCONCLUSIVE
986,debt_issuance,covered_call,71,42,1,0.697730,NaN,NaN,INCONCLUSIVE
990,debt_issuance,protective_put,70,42,1,0.103430,NaN,NaN,INCONCLUSIVE
994,debt_issuance,collar,70,42,1,0.898780,NaN,NaN,INCONCLUSIVE
998,debt_issuance,cash_secured_put,73,44,1,-0.074847,NaN,NaN,INCONCLUSIVE


## Interpretation and remaining validation

A discovery signal is not an established trading edge. Review entry premiums, stock movements and tails, liquidity, costs, outliers, event text, and stability before a frozen validation test. Synthetic stock and last-trade option prices are screening limitations. No assignment or financing model is supplied. A conclusive claim requires valid uncertainty and independent validation.
